# 02. Baseline на application_train

## Цель

Кратко:
- сравнить Logistic Regression и CatBoost только на данных заявки;
- использовать фиксированный train/holdout и библиотечную CV;
- сохранить две модели и простую таблицу сопоставимых CV-метрик.

Процесс показан явно: данные → признаки → разделение → preprocessing →
модель → CV → финальный `.fit()`. Holdout здесь не оценивается: с теми же
параметрами разделения он воспроизводится и оценивается в ноутбуке 08.


## 1. Импорты и настройки

In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import joblib
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.compose import (
    ColumnTransformer,
    make_column_selector,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import (
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)
from src.validation import validate_application_data


APPLICATION_PATH = find_data_file("application_train.csv")
CLIENT_SPLIT_PATH = PROCESSED_DATA_DIR / "client_split.csv"
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка данных и основные проверки

Проверяются уникальность `SK_ID_CURR`, бинарность `TARGET` и техническое
значение `365243` в `DAYS_EMPLOYED`.


In [4]:
application = pd.read_csv(APPLICATION_PATH)
validate_application_data(application)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

print("Размер application_train:", application.shape)
print("Доля TARGET=1:", round(application["TARGET"].mean(), 4))

Размер application_train: (307511, 122)
Доля TARGET=1: 0.0807


## 3. Единое разделение клиентов


In [5]:
if CLIENT_SPLIT_PATH.exists():
    client_split = pd.read_csv(CLIENT_SPLIT_PATH)
else:
    _, holdout_client_ids = train_test_split(
        application["SK_ID_CURR"],
        test_size=0.20,
        stratify=application["TARGET"],
        random_state=RANDOM_STATE,
    )

    client_split = application[["SK_ID_CURR"]].copy()
    client_split["split"] = "train"
    client_split.loc[
        client_split["SK_ID_CURR"].isin(holdout_client_ids),
        "split",
    ] = "holdout"
    client_split.to_csv(
        CLIENT_SPLIT_PATH,
        index=False,
    )
    print("Создан split:", CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = application.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


Создан split: /content/drive/MyDrive/credit-scoring-system/data/processed/client_split.csv
split
train      246008
holdout     61503
Name: count, dtype: int64


## 4. Формирование train-части


In [6]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 120)
Holdout clients (не используется): 61503


## 5. Preprocessing для Logistic Regression

- Числовые пропуски заменяются медианой, затем признаки масштабируются.
- Категориальные пропуски заменяются наиболее частым значением.
- Категории кодируются через `OneHotEncoder(handle_unknown="ignore")`.
- Типы колонок автоматически выбирает sklearn через
  `make_column_selector`.

Preprocessing находится внутри общего `Pipeline`. При CV он обучается
только на train-части каждого fold и не видит validation или holdout.


In [7]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

In [8]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            make_column_selector(
                dtype_include=np.number,
            ),
        ),
        (
            "categorical",
            categorical_pipeline,
            make_column_selector(
                dtype_exclude=np.number,
            ),
        ),
    ]
)


## 6. Модель 1. Logistic Regression


### Создание модели

In [10]:
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

### Объединение preprocessing и модели

In [11]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logistic_model),
    ]
)

### Кросс-валидация Logistic Regression

`cross_validate` обучает весь Pipeline отдельно на каждом fold. Поэтому
imputer, scaler и encoder не получают информацию из validation-части.

In [12]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


In [13]:
logistic_cv_results = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_splitter,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision",
    },
    return_train_score=False,
    n_jobs=-1,
)

### Метрики Logistic Regression

In [14]:
logistic_roc_auc_folds = logistic_cv_results["test_roc_auc"]
logistic_pr_auc_folds = logistic_cv_results["test_pr_auc"]

logistic_cv_roc_auc = logistic_roc_auc_folds.mean()
logistic_cv_pr_auc = logistic_pr_auc_folds.mean()
logistic_cv_roc_auc_std = logistic_roc_auc_folds.std()
logistic_cv_pr_auc_std = logistic_pr_auc_folds.std()

print("ROC-AUC по фолдам:", np.round(logistic_roc_auc_folds, 4))
print(f"Средний CV ROC-AUC: {logistic_cv_roc_auc:.4f}")
print("PR-AUC по фолдам:", np.round(logistic_pr_auc_folds, 4))
print(f"Средний CV PR-AUC: {logistic_cv_pr_auc:.4f}")

ROC-AUC по фолдам: [0.7426 0.7482 0.7437]
Средний CV ROC-AUC: 0.7448
PR-AUC по фолдам: [0.2119 0.2246 0.2172]
Средний CV PR-AUC: 0.2179


### Финальное обучение Logistic Regression

После CV Pipeline явно обучается на всей обучающей части. Holdout при этом
не используется.

In [15]:
logistic_pipeline.fit(
    X_train,
    y_train,
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c5a91d0f860>),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c5a91d0f440>)])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [16]:
joblib.dump(
    logistic_pipeline,
    MODELS_DIR / "logistic_regression_baseline.joblib",
)

['/content/drive/MyDrive/credit-scoring-system/models/logistic_regression_baseline.joblib']

## 7. Модель 2. CatBoost

CatBoost не использует sklearn-preprocessor, масштабирование или
One-Hot Encoding. Категориальные пропуски заменяются строкой, а числовые
`NaN` остаются для встроенной обработки CatBoost.


### Подготовка данных для CatBoost

In [17]:
X_train_catboost = X_train.copy()

catboost_categorical_columns = (
    X_train_catboost
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)
X_train_catboost[catboost_categorical_columns] = (
    X_train_catboost[catboost_categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

assert "TARGET" not in X_train_catboost.columns
assert "SK_ID_CURR" not in X_train_catboost.columns
print(
    "Категориальных признаков CatBoost:",
    len(catboost_categorical_columns),
)


Категориальных признаков CatBoost: 16


### Pool для CatBoost

In [18]:
catboost_train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=catboost_categorical_columns,
)


### Параметры CatBoost

In [19]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    **catboost_device_config,
}


### Кросс-валидация CatBoost

`CatBoostClassifier` описывает одну модель и поэтому не принимает
кросс-валидацию как параметр конструктора.

Для кросс-валидации используется отдельная функция `catboost.cv()`.
Она обучает модели на разных фолдах и возвращает средние метрики и их
стандартные отклонения. Объект `StratifiedKFold` передаётся в `folds`.

In [20]:
catboost_cv_results = catboost_cv(
    pool=catboost_train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.6973384	best: 0.6973384 (0)	total: 167ms	remaining: 2m 47s
100:	test: 0.7425163	best: 0.7425163 (100)	total: 11s	remaining: 1m 37s
200:	test: 0.7441475	best: 0.7441670 (193)	total: 19.9s	remaining: 1m 19s
300:	test: 0.7461169	best: 0.7461520 (294)	total: 29.6s	remaining: 1m 8s
400:	test: 0.7485873	best: 0.7485873 (398)	total: 39.5s	remaining: 59s
500:	test: 0.7494860	best: 0.7494920 (481)	total: 48.2s	remaining: 48s
600:	test: 0.7498263	best: 0.7498306 (593)	total: 57.3s	remaining: 38s
700:	test: 0.7506215	best: 0.7506297 (696)	total: 1m 7s	remaining: 28.6s
800:	test: 0.7510368	best: 0.7510368 (800)	total: 1m 16s	remaining: 19s
900:	test: 0.7512792	best: 0.7512792 (899)	total: 1m 25s	remaining: 9.44s
999:	test: 0.7516600	best: 0.7516600 (998)	total: 1m 35s	remaining: 0us
bestTest = 0.751660049
bestIteration = 998
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7043935	best: 0.7043935 (0)	total: 163ms	remaining: 2m 42s
100:	test: 0.7452846	best: 0.7452846 (99)	total: 10s	remaining: 1m 29s
200:	test: 0.7484327	best: 0.7484327 (199)	total: 18.8s	remaining: 1m 14s
300:	test: 0.7511102	best: 0.7511110 (295)	total: 28.6s	remaining: 1m 6s
400:	test: 0.7527742	best: 0.7527742 (400)	total: 37.2s	remaining: 55.5s
500:	test: 0.7532732	best: 0.7532732 (498)	total: 46.2s	remaining: 46s
600:	test: 0.7544901	best: 0.7544901 (599)	total: 56.1s	remaining: 37.2s
700:	test: 0.7551264	best: 0.7551266 (697)	total: 1m 4s	remaining: 27.4s
800:	test: 0.7554572	best: 0.7554689 (799)	total: 1m 13s	remaining: 18.4s
900:	test: 0.7560527	best: 0.7560555 (895)	total: 1m 23s	remaining: 9.21s
999:	test: 0.7562214	best: 0.7562214 (999)	total: 1m 32s	remaining: 0us
bestTest = 0.756221354
bestIteration = 999
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7060657	best: 0.7060657 (0)	total: 217ms	remaining: 3m 36s
100:	test: 0.7445539	best: 0.7445539 (99)	total: 9.81s	remaining: 1m 27s
200:	test: 0.7473629	best: 0.7473629 (199)	total: 19.7s	remaining: 1m 18s
300:	test: 0.7483692	best: 0.7483692 (299)	total: 27.8s	remaining: 1m 4s
400:	test: 0.7503627	best: 0.7503684 (399)	total: 37.7s	remaining: 56.3s
500:	test: 0.7524589	best: 0.7524589 (500)	total: 47.7s	remaining: 47.5s
600:	test: 0.7533651	best: 0.7533651 (600)	total: 56s	remaining: 37.2s
700:	test: 0.7538247	best: 0.7538761 (680)	total: 1m 6s	remaining: 28.2s
800:	test: 0.7541237	best: 0.7541775 (789)	total: 1m 16s	remaining: 18.9s
900:	test: 0.7545229	best: 0.7545246 (892)	total: 1m 24s	remaining: 9.28s
999:	test: 0.7546476	best: 0.7546476 (992)	total: 1m 34s	remaining: 0us
bestTest = 0.7546476126
bestIteration = 992


### Результат кросс-валидации

In [21]:
display(catboost_cv_results.tail())
print(catboost_cv_results.columns.tolist())

,iterations,test-AUC-mean,test-AUC-std,test-Logloss-mean,test-Logloss-std,train-Logloss-mean,train-Logloss-std,test-PRAUC:type=Classic-mean,test-PRAUC:type=Classic-std,train-PRAUC:type=Classic-mean,train-PRAUC:type=Classic-std
196,980,0.754101,0.002302,0.588763,0.001961,0.569362,0.001171,NaN,NaN,NaN,NaN
197,985,0.754118,0.002310,0.588751,0.001969,0.569201,0.001057,NaN,NaN,NaN,NaN
198,990,0.754151,0.002319,0.588724,0.001978,0.569084,0.001047,NaN,NaN,NaN,NaN
199,995,0.754172,0.002316,0.588709,0.001968,0.568992,0.001002,NaN,NaN,NaN,NaN
200,999,0.754176,0.002317,0.588704,0.001967,0.568954,0.000960,NaN,NaN,NaN,NaN


['iterations', 'test-AUC-mean', 'test-AUC-std', 'test-Logloss-mean', 'test-Logloss-std', 'train-Logloss-mean', 'train-Logloss-std', 'test-PRAUC:type=Classic-mean', 'test-PRAUC:type=Classic-std', 'train-PRAUC:type=Classic-mean', 'train-PRAUC:type=Classic-std']


### Лучшая итерация и CV-метрики CatBoost

In [22]:
auc_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)
auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)
prauc_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)
prauc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[auc_column].idxmax()
best_cv_row = catboost_cv_results.loc[best_cv_index]

catboost_best_iteration = int(best_cv_row["iterations"]) + 1
catboost_cv_roc_auc = float(best_cv_row[auc_column])
catboost_cv_roc_auc_std = float(best_cv_row[auc_std_column])
catboost_cv_pr_auc = float(best_cv_row[prauc_column])
catboost_cv_pr_auc_std = float(best_cv_row[prauc_std_column])

print(f"Лучшая итерация: {catboost_best_iteration}")
print(
    f"Средний CV ROC-AUC: {catboost_cv_roc_auc:.4f} "
    f"± {catboost_cv_roc_auc_std:.4f}"
)
print(
    f"Средний CV PR-AUC: {catboost_cv_pr_auc:.4f} "
    f"± {catboost_cv_pr_auc_std:.4f}"
)

Лучшая итерация: 1000
Средний CV ROC-AUC: 0.7542 ± 0.0023
Средний CV PR-AUC: nan ± nan


### Итоговая модель CatBoost

In [23]:
final_catboost_model = CatBoostClassifier(
    iterations=catboost_best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


### Финальное обучение CatBoost

In [24]:
final_catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=catboost_categorical_columns,
)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 62.2ms	remaining: 1m 2s
100:	total: 6.31s	remaining: 56.2s
200:	total: 10.7s	remaining: 42.5s
300:	total: 15s	remaining: 34.9s
400:	total: 21.1s	remaining: 31.5s
500:	total: 25.4s	remaining: 25.3s
600:	total: 29.8s	remaining: 19.8s
700:	total: 35.9s	remaining: 15.3s
800:	total: 40.2s	remaining: 9.99s
900:	total: 44.5s	remaining: 4.89s
999:	total: 50.3s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [25]:
final_catboost_model.save_model(
    str(MODELS_DIR / "catboost_application_baseline.cbm")
)

## 8. Единый реестр CV-метрик


In [26]:
logistic_result = {
    "experiment": "application_logistic",
    "notebook": "02_application_baseline.ipynb",
    "model": "LogisticRegression",
    "feature_set": "application",
    "source_tables": "application_train.csv",
    "device": "CPU",
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": np.nan,
    "cv_roc_auc": float(logistic_cv_roc_auc),
    "cv_roc_auc_std": float(logistic_cv_roc_auc_std),
    "cv_pr_auc": float(logistic_cv_pr_auc),
    "cv_pr_auc_std": float(logistic_cv_pr_auc_std),
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(logistic_result)

catboost_result = {
    "experiment": "application_catboost",
    "notebook": "02_application_baseline.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application",
    "source_tables": "application_train.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": catboost_best_iteration,
    "cv_roc_auc": catboost_cv_roc_auc,
    "cv_roc_auc_std": catboost_cv_roc_auc_std,
    "cv_pr_auc": catboost_cv_pr_auc,
    "cv_pr_auc_std": catboost_cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(catboost_result)
baseline_results = all_results[
    all_results["experiment"].isin(
        ["application_logistic", "application_catboost"]
    )
].copy()
display(baseline_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN


## Выводы

In [27]:
best_baseline = baseline_results.loc[
    baseline_results["cv_roc_auc"].idxmax()
]
print(
    f"Лучшая baseline-модель: {best_baseline['model']}; "
    f"CV ROC-AUC={best_baseline['cv_roc_auc']:.4f}; "
    f"CV PR-AUC={best_baseline['cv_pr_auc']:.4f}."
)
print(
    f"Для CatBoost выбрана итерация "
    f"{catboost_best_iteration} по максимуму средней test AUC."
)
print(
    "Split сохранён в client_split.csv. Holdout не открывался "
    "и не использовался для выбора baseline-модели."
)


Лучшая baseline-модель: CatBoostClassifier; CV ROC-AUC=0.7542; CV PR-AUC=nan.
Для CatBoost выбрана итерация 1000 по максимуму средней test AUC.
Split сохранён в client_split.csv. Holdout не открывался и не использовался для выбора baseline-модели.
